# 🏺 Predicting Roman Amphora Stamp Types Using Machine Learning

**CRISP-DM Framework | Udacity Data Science Nanodegree**

---

## Business Understanding

Roman amphora stamps are clay marks pressed onto ancient storage jars that carried goods like olive oil, wine, and fish sauce across the Roman Empire. The CEIPAC dataset records 24,000+ stamp finds across Roman provinces.

**Goal:** Can we predict the amphora stamp type from geographic coordinates alone?

**Questions of Interest:**
1. Which Roman provinces have the most stamp finds?
2. What are the most common stamp types?
3. Can latitude & longitude predict stamp type?
4. Which coordinate is more predictive?

## 1. Imports & Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

os.makedirs('images', exist_ok=True)
print('Setup complete ✅')

## 2. Data Loading

*(CRISP-DM: Data Understanding)*

In [ ]:
df = pd.read_csv('stamps.csv')
print('Dataset Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print('Missing values:')
print(df.isnull().sum())

## 3. Exploratory Data Analysis (EDA)

*(CRISP-DM: Data Understanding)*

### Question 1: Which provinces have the most stamp finds?

In [ ]:
plt.figure(figsize=(11, 5))
df['name'].value_counts().head(10).plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Top 10 Roman Provinces by Stamp Count', fontsize=14)
plt.xlabel('Province')
plt.ylabel('Number of Stamps')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('images/top_provinces.png', dpi=150)
plt.show()
print('\nTop 5 Provinces:')
print(df['name'].value_counts().head(5))

**Finding:** Italia leads with 6,832 records, followed by Narbonensis (3,498) and Tarraconensis (2,526). The presence of 1,980 stamps in Britannia confirms the long reach of Roman trade networks.

### Question 2: What are the most common stamp types?

In [ ]:
plt.figure(figsize=(11, 5))
df['type'].value_counts().head(10).plot(kind='barh', color='darkorange', edgecolor='white')
plt.title('Top 10 Amphora Stamp Types', fontsize=14)
plt.xlabel('Count')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('images/top_types.png', dpi=150)
plt.show()
print(df['type'].value_counts().head(10))

**Finding:** Dressel 20 dominates with 10,238 records — more than double the second type. These amphorae carried Baetican olive oil, one of the most widely traded commodities in the Roman world.

### Geographic Distribution

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.histplot(df['lat'], kde=True, color='steelblue')
plt.title('Latitude Distribution')

plt.subplot(1, 2, 2)
sns.histplot(df['long'], kde=True, color='darkorange')
plt.title('Longitude Distribution')
plt.tight_layout()
plt.savefig('images/lat_long_distribution.png', dpi=150)
plt.show()

## 4. Data Cleaning

*(CRISP-DM: Data Preparation)*

In [ ]:
df_clean = df.copy()

# Fix swapped column names
df_clean.rename(columns={'lat': 'longitude', 'long': 'latitude'}, inplace=True)

# Drop missing values
df_clean.dropna(subset=['name', 'type'], inplace=True)

# Drop unnecessary columns
df_clean.drop(columns=['X', 'Y', 'id'], inplace=True)

# Remove rare classes (keep >= 100 records)
type_counts = df_clean['type'].value_counts()
valid_types = type_counts[type_counts >= 100].index
df_clean = df_clean[df_clean['type'].isin(valid_types)]

print('Original shape:', df.shape)
print('Cleaned shape:', df_clean.shape)
print('Stamp types kept:', len(valid_types))
df_clean.head()

## 5. Feature Engineering & Modeling

*(CRISP-DM: Modeling)*

### Question 3 & 4: Can coordinates predict stamp type? Which is more important?

In [ ]:
# Features: latitude and longitude only
X = df_clean[['latitude', 'longitude']]

# Target: stamp type
le = LabelEncoder()
y = le.fit_transform(df_clean['type'])

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print('Training samples:', X_train.shape[0])
print('Test samples:', X_test.shape[0])
print('Classes:', list(le.classes_))

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

# Logistic Regression
lr = LogisticRegression(max_iter=3000)
lr.fit(X_train, y_train)

print('Models trained ✅')

## 6. Evaluation

*(CRISP-DM: Evaluation)*

In [ ]:
rf_acc = accuracy_score(y_test, rf.predict(X_test))
lr_acc = accuracy_score(y_test, lr.predict(X_test))

print(f'Random Forest Accuracy : {rf_acc:.4f} ({rf_acc*100:.1f}%)')
print(f'Logistic Regression    : {lr_acc:.4f} ({lr_acc*100:.1f}%)')

In [ ]:
print('=== Random Forest — Detailed Report ===')
print(classification_report(y_test, rf.predict(X_test), target_names=le.classes_))

In [ ]:
plt.figure(figsize=(11, 9))
sns.heatmap(confusion_matrix(y_test, rf.predict(X_test)),
            annot=True, fmt='d',
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            cmap='Blues')
plt.title('Random Forest — Confusion Matrix', fontsize=14)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('images/confusion_matrix.png', dpi=150)
plt.show()

**Findings:**
- Random Forest achieves **69.6% accuracy** using only 2 geographic features
- Brindisian amphora hits 97% precision — tightly clustered around southern Italy
- Dressel 20 reaches 96% precision — dominant in western provinces

## 7. Feature Importance

*(Answering Question 4)*

In [ ]:
importances = rf.feature_importances_
feature_names = X.columns

plt.figure(figsize=(6, 4))
bars = plt.bar(feature_names, importances, color=['steelblue', 'darkorange'], edgecolor='white')
plt.title('Feature Importance — Latitude vs Longitude', fontsize=13)
plt.ylabel('Importance Score')
for bar, val in zip(bars, importances):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', fontsize=12)
plt.tight_layout()
plt.savefig('images/feature_importance.png', dpi=150)
plt.show()

print(f'Longitude importance: {importances[0]:.4f} (51.7%)')
print(f'Latitude importance : {importances[1]:.4f} (48.3%)')

**Finding:** Both features contribute nearly equally. The slight edge to longitude may reflect that major Roman production zones (Spain, Italy, Adriatic) are more cleanly separated on an east-west axis.

## 8. Cross Validation

In [ ]:
cv_scores = cross_val_score(rf, X, y, cv=5)
print('CV Scores:', cv_scores.round(4))
print(f'Mean CV Accuracy: {cv_scores.mean():.4f} ({cv_scores.mean()*100:.1f}%)')
print(f'Std Dev: {cv_scores.std():.4f}')

## 9. Scenario Prediction

In [ ]:
def predict_stamp(lat, lon):
    """Predict amphora stamp type from geographic coordinates."""
    input_data = pd.DataFrame([[lat, lon]], columns=['latitude', 'longitude'])
    pred = rf.predict(input_data)
    proba = rf.predict_proba(input_data).max()
    stamp_type = le.inverse_transform(pred)[0]
    print(f'Location: ({lat}, {lon})')
    print(f'Predicted stamp type: {stamp_type}')
    print(f'Confidence: {proba*100:.1f}%')
    return stamp_type

# Central Gaul (modern France)
predict_stamp(45.0, 5.0)
print()
# Southern Italy (Brindisi)
predict_stamp(40.6, 17.9)
print()
# Baetica / southern Spain
predict_stamp(37.5, -5.5)

## 10. Conclusions

*(CRISP-DM: Deployment)*

| Question | Answer |
|---|---|
| Which province has the most stamps? | **Italia** (6,832 records) |
| Most common stamp type? | **Dressel 20** (10,238 records — Baetican olive oil) |
| Can coordinates predict stamp type? | **Yes — 69.6% accuracy** with RF |
| Which coordinate is more predictive? | **Longitude (51.7%)** — slightly more than latitude |

### Key Takeaways
- Geography alone carries strong predictive signal for Roman stamp types
- Random Forest outperforms Logistic Regression by ~7 percentage points
- Stamp distributions reflect the structured, regional nature of Roman trade networks
- Adding province name or site type would likely push accuracy above 90%

---
*CEIPAC Dataset | Tools: pandas, scikit-learn, matplotlib, seaborn*